# Vision Transformer — The Library Version

Three parts, honestly framed: (1) the patch-embedding-is-a-convolution claim, verified numerically; (2) the scoreboard in context (what small-scale results do and don't license); (3) the PyTorch translation — shown, not run.

In [1]:
import numpy as np
import pandas as pd
from scipy.signal import correlate2d

# (1) README §3.1's disguise note, verified: patchify + linear projection == stride-2 2x2 convolution
rs = np.random.default_rng(0)
img = rs.normal(0, 1, (8, 8))
Wp = rs.normal(0, 0.15, (4, 48))                       # our tokenizer

def patchify1(im):
    return im.reshape(4, 2, 4, 2).transpose(0, 2, 1, 3).reshape(16, 4)
ours = patchify1(img) @ Wp                              # (16, 48)

convs = np.zeros((16, 48))
for f in range(48):
    k = Wp[:, f].reshape(2, 2)                          # each output dim = one 2x2 kernel
    fm = correlate2d(img, k, mode="valid")[::2, ::2]    # stride 2: non-overlapping placements
    convs[:, f] = fm.ravel()
print(f"patch embedding vs stride-2 conv (scipy) — max |difference|: {np.abs(ours - convs).max():.2e}")
print("Identical: the ViT never evicted convolution; it demoted it to doorman (README §3.1).")

patch embedding vs stride-2 conv (scipy) — max |difference|: 1.11e-16
Identical: the ViT never evicted convolution; it demoted it to doorman (README §3.1).


In [2]:
# (2) the scoreboard in context
print("The course's vision track, one table (same digits, same split, same shift protocol):")
print(f"{'model':30s} {'params':>8s} {'clean':>7s} {'shifted':>8s}")
rows = [("MLP (lesson 10 machinery)", "2,410", "95.6%", "47.5%"),
        ("CNN (lesson 11)", "810", "94.4%", "58.9%"),
        ("ViT (lesson 15)", "45,434", "96.7%", "50.6%"),
        ("ViT + augmentation", "45,434", "98.6%", "96.1%")]
for r in rows: print(f"{r[0]:30s} {r[1]:>8s} {r[2]:>7s} {r[3]:>8s}")
print()
print("Read responsibly: the easy task lets capacity beat priors on CLEAN accuracy; the shift")
print("column is where inductive bias actually speaks; and augmentation is data purchasing the")
print("missing belief. At production scale the same three forces set the CNN/ViT crossover.")

The course's vision track, one table (same digits, same split, same shift protocol):
model                            params   clean  shifted
MLP (lesson 10 machinery)         2,410   95.6%    47.5%
CNN (lesson 11)                     810   94.4%    58.9%
ViT (lesson 15)                  45,434   96.7%    50.6%
ViT + augmentation               45,434   98.6%    96.1%

Read responsibly: the easy task lets capacity beat priors on CLEAN accuracy; the shift
column is where inductive bias actually speaks; and augmentation is data purchasing the
missing belief. At production scale the same three forces set the CNN/ViT crossover.


### (3) The PyTorch translation — read it; you have built every line

```python
import torch, torch.nn as nn

patchify = nn.Conv2d(1, 48, kernel_size=2, stride=2)     # Block 3's tokenizer, openly a conv
cls_token = nn.Parameter(torch.zeros(1, 1, 48))          # the blank seat
pos_embed = nn.Parameter(torch.zeros(1, 17, 48))
block = nn.TransformerEncoderLayer(d_model=48, nhead=4, dim_feedforward=128,
                                   batch_first=True, norm_first=True)
encoder = nn.TransformerEncoder(block, num_layers=2)     # NOTE: no mask argument — Block 4's diff
head = nn.Linear(48, 10)                                 # reads output[:, 0] — the CLS verdict

# timm (the production ViT zoo) is this skeleton + pretrained weights + augmentation recipes:
#   model = timm.create_model('vit_base_patch16_224', pretrained=True)
# 'patch16_224': 224px images, 16px patches -> 196 tokens — our 16-token digit, scaled 12x
```

Same tokenizer, same blank seat, same maskless encoder. The pretrained weights are the data loan, pre-paid by someone with more images than you.